# Summary Tables — Sector Returns by Macro Regime

This notebook produces tabular summaries of sector performance across macro regimes:

1. **Average daily returns** by regime
2. **Annualized returns** by regime
3. **Sector rankings** within each regime
4. **Best / worst sector** quick-reference per regime
5. **Observation counts** per regime

In [ ]:
import pandas as pd

SECTOR_NAMES = {
    'XLB': 'Materials',
    'XLC': 'Comm Svcs',
    'XLE': 'Energy',
    'XLF': 'Financials',
    'XLI': 'Industrials',
    'XLK': 'Technology',
    'XLP': 'Cons Staples',
    'XLRE': 'Real Estate',
    'XLU': 'Utilities',
    'XLV': 'Health Care',
    'XLY': 'Cons Discr',
}

# load pivot tables -- avg daily returns
df_combined   = pd.read_csv('../data/processed/pivot_sector_by_combined_regime.csv',   index_col=0)
df_vix        = pd.read_csv('../data/processed/pivot_sector_by_vix_regime.csv',        index_col=0)
df_inflation  = pd.read_csv('../data/processed/pivot_sector_by_inflation_regime.csv',  index_col=0)
df_rate       = pd.read_csv('../data/processed/pivot_sector_by_rate_regime.csv',       index_col=0)

#load annualized return tables
df_ann_combined  = pd.read_csv('../data/processed/annualized_return_by_combined_regime.csv',  index_col=0)
df_ann_vix       = pd.read_csv('../data/processed/annualized_return_by_vix_regime.csv',       index_col=0)
df_ann_inflation = pd.read_csv('../data/processed/annualized_return_by_inflation_regime.csv', index_col=0)
df_ann_rate      = pd.read_csv('../data/processed/annualized_return_by_rate_regime.csv',      index_col=0)

#long-form summary (includes observation counts)
df_summary = pd.read_csv('../data/processed/avg_return_summary.csv')

print('Data loaded successfully.')

## Helper Functions

In [ ]:
def styled_pct(df, caption, cmap='RdYlGn'):
    """Style a returns DataFrame as color-coded percentages."""
    out = df.rename(columns=SECTOR_NAMES) * 100
    return (
        out.style
        .format('{:.2f}%', na_rep='—')
        .background_gradient(cmap=cmap, axis=None)
        .set_caption(caption)
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size','13px'),('font-weight','bold'),('text-align','left'),('padding-bottom','6px')]},
            {'selector': 'th',      'props': [('text-align','center'),('font-size','11px')]},
            {'selector': 'td',      'props': [('text-align','center'),('font-size','11px')]},
        ])
    )


def styled_rank(df, caption):
    """Rank sectors within each regime row (1 = best). Color green→red."""
    out = df.rename(columns=SECTOR_NAMES)
    ranks = out.rank(axis=1, ascending=False, na_option='bottom').astype('Int64')
    n = out.shape[1]
    return (
        ranks.style
        .format('{}')
        .background_gradient(cmap='RdYlGn_r', axis=None, vmin=1, vmax=n)
        .set_caption(caption)
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size','13px'),('font-weight','bold'),('text-align','left'),('padding-bottom','6px')]},
            {'selector': 'th',      'props': [('text-align','center'),('font-size','11px')]},
            {'selector': 'td',      'props': [('text-align','center'),('font-size','11px')]},
        ])
    )

---
## Table 1: Average Daily Returns by Regime (%)

In [ ]:
display(styled_pct(df_combined,  'Combined Macro Regime — Avg Daily Return (%)'))
display(styled_pct(df_vix,       'VIX Regime — Avg Daily Return (%)'))
display(styled_pct(df_inflation,  'Inflation Regime — Avg Daily Return (%)'))
display(styled_pct(df_rate,      'Interest Rate Regime — Avg Daily Return (%)'))

---
## Table 2: Annualized Returns by Regime (%)

In [ ]:
display(styled_pct(df_ann_combined,  'Combined Macro Regime — Annualized Return (%)'))
display(styled_pct(df_ann_vix,       'VIX Regime — Annualized Return (%)'))
display(styled_pct(df_ann_inflation,  'Inflation Regime — Annualized Return (%)'))
display(styled_pct(df_ann_rate,      'Interest Rate Regime — Annualized Return (%)'))

---
## Table 3: Sector Rankings within Each Regime

Rank 1 = best-performing sector. Green = top, red = bottom.

In [ ]:
display(styled_rank(df_combined,  'Combined Macro Regime — Sector Rank (1 = Best)'))
display(styled_rank(df_vix,       'VIX Regime — Sector Rank (1 = Best)'))
display(styled_rank(df_inflation,  'Inflation Regime — Sector Rank (1 = Best)'))
display(styled_rank(df_rate,      'Interest Rate Regime — Sector Rank (1 = Best)'))

---
## Table 4: Best & Worst Sector per Regime

Quick-reference showing the top and bottom sector for each regime across all regime types.

In [ ]:
def best_worst(df, regime_type_label):
    rows = []
    for regime, row in df.iterrows():
        valid = row.dropna()
        if valid.empty:
            continue
        best_ticker = valid.idxmax()
        worst_ticker = valid.idxmin()
        rows.append({
            'Regime Type': regime_type_label,
            'Regime': regime,
            'Best Sector': SECTOR_NAMES.get(best_ticker, best_ticker),
            'Best Return': f"{valid[best_ticker]*100:.2f}%",
            'Worst Sector': SECTOR_NAMES.get(worst_ticker, worst_ticker),
            'Worst Return': f"{valid[worst_ticker]*100:.2f}%",
        })
    return rows

rows = (
    best_worst(df_combined,  'Combined')
    + best_worst(df_vix,       'VIX')
    + best_worst(df_inflation,  'Inflation')
    + best_worst(df_rate,      'Rate')
)

bw = pd.DataFrame(rows).set_index(['Regime Type', 'Regime'])
display(
    bw.style
    .set_caption('Best & Worst Sector per Regime (Avg Daily Return)')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size','13px'),('font-weight','bold'),('text-align','left'),('padding-bottom','6px')]},
        {'selector': 'th',      'props': [('text-align','center'),('font-size','11px')]},
        {'selector': 'td',      'props': [('text-align','center'),('font-size','11px')]},
    ])
)

---
## Table 5: Observation Counts per Regime

Number of daily return observations underlying each regime cell.

In [ ]:
obs = (
    df_summary
    .groupby(['regime_type', 'regime_value'])['n_observations']
    .agg(['min', 'max', 'mean'])
    .rename(columns={'min': 'Min Obs', 'max': 'Max Obs', 'mean': 'Avg Obs'})
    .round(0)
    .astype(int)
)

obs.index.names = ['Regime Type', 'Regime Value']

display(
    obs.style
    .format('{:,}')
    .background_gradient(cmap='Blues', subset=['Avg Obs'])
    .set_caption('Observation Counts per Regime (min / max / avg across sectors)')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size','13px'),('font-weight','bold'),('text-align','left'),('padding-bottom','6px')]},
        {'selector': 'th',      'props': [('text-align','center'),('font-size','11px')]},
        {'selector': 'td',      'props': [('text-align','center'),('font-size','11px')]},
    ])
)